# Тюнинг гиперпараметров

In [ ]:
cat_features = [
    'researchDayType',
    'programAgeRestrictionName',
    'breaksPrimeTimeStatusName',
    'day_of_week',
    'month',
    'season'
]

# Копируем X_train/X_test, чтобы избежать SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()

# Приводим к category
for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')


## Тюнинг CatBoost

In [ ]:
# Целевая функция для Optuna
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 300),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.2),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1.0, 10.0),
        'loss_function': 'RMSE',
        'eval_metric': 'R2',
        'cat_features': cat_features,
        'verbose': 0,
        'random_seed': 42,
        'task_type': 'CPU'
    }

    model = CatBoostRegressor(**params)


    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=3,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

# Запуск Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print("Лучшие параметры:", study.best_params)

[I 2025-05-03 20:06:31,536] A new study created in memory with name: no-name-fa023677-40d7-4018-bbcb-c7051d703a46
[I 2025-05-03 20:29:07,988] Trial 0 finished with value: 0.9454635390124081 and parameters: {'iterations': 286, 'depth': 7, 'learning_rate': 0.04484785903089612, 'l2_leaf_reg': 6.689580354767562, 'bagging_temperature': 0.5500143761162274, 'random_strength': 2.005259183962843}. Best is trial 0 with value: 0.9454635390124081.
[I 2025-05-03 20:34:09,411] Trial 1 finished with value: 0.9468764192927157 and parameters: {'iterations': 175, 'depth': 7, 'learning_rate': 0.11165590122147204, 'l2_leaf_reg': 2.387272315860455, 'bagging_temperature': 0.7603891405820202, 'random_strength': 9.185368513608136}. Best is trial 1 with value: 0.9468764192927157.
[I 2025-05-03 20:37:06,938] Trial 2 finished with value: 0.9406166257537706 and parameters: {'iterations': 127, 'depth': 4, 'learning_rate': 0.1815217885581106, 'l2_leaf_reg': 6.687198729104921, 'bagging_temperature': 0.56517812506234

Лучшие параметры: {'iterations': 175, 'depth': 7, 'learning_rate': 0.11165590122147204, 'l2_leaf_reg': 2.387272315860455, 'bagging_temperature': 0.7603891405820202, 'random_strength': 9.185368513608136}


In [ ]:
# Обучаем CatBoost с лучшими параметрами
best_catboost = CatBoostRegressor(
    **study.best_params,
    loss_function='RMSE',
    eval_metric='R2',
    cat_features=cat_features,
    verbose=100,
    random_seed=42
)

best_catboost.fit(X_train, y_train)

0:	learn: 0.1777095	total: 836ms	remaining: 2m 25s
100:	learn: 0.9451566	total: 1m 23s	remaining: 1m
174:	learn: 0.9510394	total: 2m 42s	remaining: 0us
